In [1]:
from __future__ import division

import numpy as np
import glob, os, json, pickle
import matplotlib.pyplot as plt
import scipy.linalg as sl

import libstempo as libs
import libstempo.plot as libsplt

import enterprise
from enterprise.pulsar import Pulsar
from enterprise.signals import parameter
from enterprise.signals import selections
from enterprise.signals import white_signals
from enterprise.signals import utils
from enterprise.signals import gp_signals
from enterprise.signals import signal_base

import corner
from PTMCMCSampler.PTMCMCSampler import PTSampler as ptmcmc

In [2]:
#NEED TO CHANGE FILE ON DIFFERENT RUNS (ie full_run_1 -> full_run_2)
runnum = '2'
dataset = 'dataset_2b'
group = 'group1'
if group == 'group1':
    closed = False
    runname = 'full_run_open_' + runnum
elif group == 'group2':
    closed = True
    runname = 'full_run_closed_' + runnum
else:
    print('Invalid group name')
    sys.exit(0)
refit = False

topdir = os.getcwd()
#Where the original data is
origdatadir = topdir + '/mdc2/' + group + '/' + dataset + '/'
#Where the json noise file is
if not closed:
    updatednoisefile = topdir + '/mdc2/' + group + '/group1_psr_noise.json'
else:
    updatednoisefile = topdir + '/' + dataset + '/fit_psr_noise.json'
#Where the dataset files are located
datadir = topdir + '/' + dataset + '/'
#Where the refit par files are
pardir = topdir + '/' + dataset + '/newpars' + dataset[-2:] + '/'
#Where the chains should be saved to
chaindir = topdir + '/' + dataset + '/chains/'
#Where the everything should be saved to (chains, cornerplts, histograms, etc.)
outdir = topdir + '/' + dataset + '/' + runname + '/'
#Where we save figures n stuff
figdir = topdir + '/' + dataset + '/Cornerplts/'
#The pickled pulsars
psr_obj_file = topdir + '/' + dataset + '/psr_objects.pickle'

if os.path.exists(datadir) == False:
    os.mkdir(datadir)
if os.path.exists(outdir) == False:
    os.mkdir(outdir)

In [3]:
def Refit_pars(origdir,newdir):
    orig_parfiles = sorted(glob.glob(origdir + '/*.par'))
    orig_timfiles = sorted(glob.glob(origdir + '/*.tim'))
    #Load all of the Pulsars into libstempo
    orig_libs_psrs = []
    for p, t in zip(orig_parfiles, orig_timfiles):
        orig_libs_psr = libs.tempopulsar(p, t)
        orig_libs_psrs.append(orig_libs_psr)

    #Fit the par files again
    #Save them to new directory (Overwrites ones currently used in newdatadir)
    if os.path.exists(newdir) == False:
        os.mkdir(newdir)
    for new_libs_psr in orig_libs_psrs:
        new_libs_psr['DM'].fit = False
        new_libs_psr['DM1'].fit = False
        new_libs_psr['DM2'].fit = False
        try:
            new_libs_psr.fit(iters=3)
        except:
            continue
        new_libs_psr.savepar(newdir + new_libs_psr.name + '.par')

In [4]:
if refit:
    #Refitting par files using libstempo
    Refit_pars(origdatadir,pardir)
    
    #Loading par and tim files into enterprise Pulsar class
    parfiles = sorted(glob.glob(pardir + '/*.par'))
    timfiles = sorted(glob.glob(origdatadir + '/*.tim'))

    #Load all the pulsars if no pickle file
    try:    #Load pulsars from pickle file
        with open(psr_obj_file,'rb') as psrfile:
            psrs = pickle.load(psrfile)
            psrfile.close()
    except:   #If no pickle file, load and save pulsars
        psrs = []
        for p, t in zip(parfiles,timfiles):
            psr = Pulsar(p, t)
            psrs.append(psr)
        #Save 9yr pulsars to a pickle file
        with open(psr_obj_file,'wb') as psrfile:
            pickle.dump(psrs,psrfile)
            psrfile.close()
else:
    parfiles = sorted(glob.glob(origdatadir + '/*.par'))
    timfiles = sorted(glob.glob(origdatadir + '/*.tim'))
    psrs = []
    for p, t in zip(parfiles,timfiles):
        psr = Pulsar(p, t)
        psrs.append(psr)

# find the maximum time span to set GW frequency sampling
tmin = [p.toas.min() for p in psrs]
tmax = [p.toas.max() for p in psrs]
Tspan = np.max(tmax) - np.min(tmin)

In [5]:
##### parameters and priors #####

# white noise parameters
efac = parameter.Uniform(0.5,3.0)
log10_equad = parameter.Uniform(-8.5,5)

# red noise parameters
red_noise_log10_A = parameter.Uniform(-20,-11)
red_noise_gamma = parameter.Uniform(0,7)

# GW parameters (initialize with names here to use parameters in common across pulsars)
log10_A_gw = parameter.Uniform(-20,-11)('zlog10_A_gw')
gamma_gw = parameter.Constant(13/3)('zgamma_gw')

##### Set up signals #####

# timing model
tm = gp_signals.TimingModel()

# white noise
ef = white_signals.MeasurementNoise(efac=efac)
eq = white_signals.EquadNoise(log10_equad = log10_equad)

# red noise (powerlaw with 30 frequencies)
pl = utils.powerlaw(log10_A=red_noise_log10_A, gamma=red_noise_gamma)
rn = gp_signals.FourierBasisGP(spectrum=pl, components=30, Tspan=Tspan)

cpl = utils.powerlaw(log10_A=log10_A_gw, gamma=gamma_gw)
# Hellings and Downs ORF
orf = utils.hd_orf()

#Common red noise process with no correlations
#crn = gp_signals.FourierBasisGP(spectrum = cpl, components=30, Tspan=Tspan, name = 'gw')

# gwb with Hellings and Downs correlations
gwb = gp_signals.FourierBasisCommonGP(cpl, orf, components=30, name='gw', Tspan=Tspan)

# full model is sum of components
model = ef + eq + rn + tm + gwb

# initialize PTA
pta = signal_base.PTA([model(psr) for psr in psrs])

In [6]:
#make dictionary of pulsar parameters from these runs
param_dict = {}
for psr in pta.pulsars:
    param_dict[psr] = {}
    for param, idx in zip(pta.param_names,range(len(pta.param_names))):
        if param.startswith(psr):
            param_dict[psr][param] = idx
print(pta.param_names)
#Save to json file
with open(outdir + '/Search_params.json','w') as paramfile:
    json.dump(param_dict,paramfile,sort_keys = True,indent = 4)
    paramfile.close()

['J0030+0451_efac', 'J0030+0451_log10_equad', 'J0030+0451_red_noise_gamma', 'J0030+0451_red_noise_log10_A', 'J0034-0534_efac', 'J0034-0534_log10_equad', 'J0034-0534_red_noise_gamma', 'J0034-0534_red_noise_log10_A', 'J0218+4232_efac', 'J0218+4232_log10_equad', 'J0218+4232_red_noise_gamma', 'J0218+4232_red_noise_log10_A', 'J0437-4715_efac', 'J0437-4715_log10_equad', 'J0437-4715_red_noise_gamma', 'J0437-4715_red_noise_log10_A', 'J0613-0200_efac', 'J0613-0200_log10_equad', 'J0613-0200_red_noise_gamma', 'J0613-0200_red_noise_log10_A', 'J0621+1002_efac', 'J0621+1002_log10_equad', 'J0621+1002_red_noise_gamma', 'J0621+1002_red_noise_log10_A', 'J0711-6830_efac', 'J0711-6830_log10_equad', 'J0711-6830_red_noise_gamma', 'J0711-6830_red_noise_log10_A', 'J0751+1807_efac', 'J0751+1807_log10_equad', 'J0751+1807_red_noise_gamma', 'J0751+1807_red_noise_log10_A', 'J0900-3144_efac', 'J0900-3144_log10_equad', 'J0900-3144_red_noise_gamma', 'J0900-3144_red_noise_log10_A', 'J1012+5307_efac', 'J1012+5307_log10

In [7]:
#Pick random initial sampling
xs = {par.name: par.sample() for par in pta.params}

# dimension of parameter space
ndim = len(pta.param_names)

# initial jump covariance matrix
cov = np.diag(np.ones(ndim) * 0.01**2)

# set up jump groups by red noise groups
groups  = [range(0, ndim)]
groups.extend(map(list, zip(range(0,ndim,2), range(1,ndim,2))))
groups.extend([[ndim-1]])

# intialize sampler
sampler = ptmcmc(ndim, pta.get_lnlikelihood, pta.get_lnprior, cov, groups=groups, outDir = outdir)
#Use this one if you want to do more runs/ RESUME is on. Be careful about saving different runs
#sampler = ptmcmc(ndim, pta.get_lnlikelihood, pta.get_lnprior, cov, groups=groups, outDir = outdir, resume = True)


In [8]:
# sampler for N steps
N = 1000000
x0 = np.hstack(p.sample() for p in pta.params)
sampler.sample(x0, N, SCAMweight=30, AMweight=15, DEweight=50)

Finished 0.10 percent in 1247.189105 s Acceptance rate = 0.828

/opt/pulsar/python/anaconda3/5.2/envs/enterprise/lib/python3.6/site-packages/enterprise/signals/parameter.py:66: RuntimeWarning: divide by zero encountered in log
  logpdf = np.log(self.prior(value, **kwargs))


Finished 1.00 percent in 8728.347450 s Acceptance rate = 0.727367Adding DE jump with weight 50
Finished 1.10 percent in 9331.916133 s Acceptance rate = 0.715545

KeyboardInterrupt: 